# Modelo preditivo de risco de defasagem

## Objetivo

Este notebook treina e avalia um modelo **realmente preditivo** para estimar o **risco de defasagem futura** do aluno, usando os indicadores observados no ano atual.

Nesta versão, a base de modelagem já foi ajustada para trabalhar com o alvo futuro:

- **`defasagem_ano_seguinte`**
- **`risco_defasagem_futuro`**

Assim, o problema de Machine Learning passa a ser:

> Dado o desempenho e o contexto do aluno no ano atual, prever a chance de ele apresentar defasagem no ano seguinte.


## Estratégia metodológica

Para tornar a modelagem mais consistente, este notebook adota as seguintes decisões:

- uso da base processada `base_processada_reduzida_ML.parquet`;
- preparação das variáveis com as funções do novo `src/features.py`;
- pipeline com imputação, padronização e codificação categórica;
- **regressão logística** como modelo baseline;
- **validação temporal preferencial**, usando o último ano disponível como teste, para respeitar a natureza preditiva do problema.

Caso a separação temporal não seja possível por falta de anos suficientes com alvo válido, o notebook usa divisão aleatória estratificada como fallback.


## Por que a regressão logística foi escolhida inicialmente

A regressão logística foi utilizada como primeiro modelo por motivos importantes para esta etapa do projeto:

- **Interpretabilidade**: permite entender com clareza como os atributos influenciam o risco previsto.
- **Baseline robusto**: é um modelo clássico e confiável para comparação com modelos mais complexos.
- **Boa adequação para classificação binária**: o alvo principal é prever `risco_defasagem_futuro` (0 ou 1).
- **Facilidade de deploy no Streamlit**: é leve, rápida e simples de serializar e publicar.


### 1) Importações e configuração do ambiente


In [ ]:
from pathlib import Path
import sys
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

ROOT_DIR = Path.cwd().resolve().parents[0] if "notebooks" in str(Path.cwd()) else Path.cwd().resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.features import (
    FEATURE_COLUMNS,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COLUMN,
    prepare_training_features,
)
from src.model import (
    load_training_data,
    build_pipeline,
    MODEL_FILE,
    METADATA_FILE,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

print("ROOT_DIR:", ROOT_DIR)
print("TARGET_COLUMN:", TARGET_COLUMN)
print("FEATURE_COLUMNS:", FEATURE_COLUMNS)

In [ ]:
from src.model import train_model

artifacts = train_model()

print("Classes:", artifacts["model"].named_steps["model"].classes_)

In [66]:
from src.model import train_model

artifacts = train_model(save_artifacts=True)

print("Classes:", artifacts["model"].named_steps["model"].classes_)
print("ROC AUC:", artifacts["metrics"].get("roc_auc"))
print(artifacts["metrics"]["classification_report"])

Classes: [0 1]
ROC AUC: 0.8965706447187929
{'0': {'precision': 0.9898477157360406, 'recall': 0.8024691358024691, 'f1-score': 0.8863636363636364, 'support': 243.0}, '1': {'precision': 0.3684210526315789, 'recall': 0.9333333333333333, 'f1-score': 0.5283018867924528, 'support': 30.0}, 'accuracy': 0.8168498168498168, 'macro avg': {'precision': 0.6791343841838098, 'recall': 0.8679012345679012, 'f1-score': 0.7073327615780446, 'support': 273.0}, 'weighted avg': {'precision': 0.9215590714388471, 'recall': 0.8168498168498168, 'f1-score': 0.847016191355814, 'support': 273.0}}


In [65]:
import pandas as pd
from src.model import load_model, predict_dataframe, predict_proba_dataframe

artifacts = load_model()
model = artifacts["model"]

cenario_baixo = pd.DataFrame([{
    "inde": 8.5,
    "n_av": 7,
    "iaa": 8,
    "ieg": 8,
    "ips": 8,
    "ipp": 8,
    "ida": 8,
    "mat": 8,
    "por": 8,
    "ing": 8,
    "ipv": 8,
    "ian": 8,
    "fase_ideal": "7",
}])

cenario_alto = pd.DataFrame([{
    "inde": 4.0,
    "n_av": 3,
    "iaa": 4,
    "ieg": 4,
    "ips": 4,
    "ipp": 4,
    "ida": 4,
    "mat": 3,
    "por": 4,
    "ing": 3,
    "ipv": 4,
    "ian": 4,
    "fase_ideal": "4",
}])

print("Baixo risco:", predict_dataframe(cenario_baixo, model=model, artifacts=artifacts))
print("Baixo risco proba:", predict_proba_dataframe(cenario_baixo, model=model, artifacts=artifacts))

print("Alto risco:", predict_dataframe(cenario_alto, model=model, artifacts=artifacts))
print("Alto risco proba:", predict_proba_dataframe(cenario_alto, model=model, artifacts=artifacts))

Baixo risco: [1]
Baixo risco proba: [[0.0083253 0.9916747]]
Alto risco: [0]
Alto risco proba: [[9.99964692e-01 3.53082007e-05]]


In [ ]:
cols = [
    "inde", "n_av", "iaa", "ieg", "ips", "ipp", "ida",
    "mat", "por", "ing", "ipv", "ian", "defasagem"
]

df_train = pd.read_parquet("../data/processed/base_processada_reduzida_ML.parquet")
df_train = df_train[df_train["risco_defasagem_futuro"].notna()].copy()

display(df_train.groupby("risco_defasagem_futuro")[cols].mean().T)
display(df_train.groupby("risco_defasagem_futuro")["fase_ideal"].value_counts().unstack(fill_value=0))

In [ ]:
import pandas as pd

df = pd.read_parquet("../data/processed/base_processada_reduzida_ML.parquet")

print(df["risco_defasagem_futuro"].value_counts(dropna=False))
print()
print(
    pd.crosstab(
        df["risco_defasagem_futuro"],
        df["defasagem_ano_seguinte"] > 0,
        dropna=False
    )
)
df.loc[:, ["defasagem_ano_seguinte", "risco_defasagem_futuro"]].dropna().head(20)

In [59]:
import pandas as pd

df = pd.read_parquet("../data/processed/base_processada_reduzida_ML.parquet")

print(df["risco_defasagem_futuro"].value_counts(dropna=False))

print(
    pd.crosstab(
        df["risco_defasagem_futuro"],
        df["defasagem_ano_seguinte"] > df["defasagem"],
        dropna=False
    )
)

risco_defasagem_futuro
<NA>    1665
0        867
1        498
Name: count, dtype: Int64
col_0                   False  True 
risco_defasagem_futuro              
0                         867      0
1                           0    498
<NA>                     1665      0


### 2) Carregar a base de treino


In [ ]:
training_path = ROOT_DIR / "data" / "processed" / "base_processada_reduzida_ML.parquet"
df = load_training_data(training_path)

print("Arquivo:", training_path)
print("Shape da base carregada:", df.shape)
display(df.head())


### 3) Entendimento inicial da base

Aqui verificamos:

- colunas disponíveis;
- tipos de dados;
- distribuição do alvo;
- anos presentes;
- valores ausentes.


In [ ]:
print("Colunas da base:")
print(df.columns.tolist())

print("\nTipos de dados:")
display(df.dtypes.to_frame("dtype"))

print("\nDistribuição da variável alvo (incluindo nulos):")
display(df[TARGET_COLUMN].value_counts(dropna=False).rename("qtd").to_frame())

print("\nDistribuição percentual do alvo (incluindo nulos):")
display((df[TARGET_COLUMN].value_counts(normalize=True, dropna=False) * 100).round(2).rename("%").to_frame())

if "ano_pede" in df.columns:
    print("\nDistribuição por ano:")
    display(df["ano_pede"].value_counts(dropna=False).sort_index().rename("qtd").to_frame())

print("\nValores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False).rename("missing").to_frame())


### 4) Preparação das features e do target

A função `prepare_training_features` do novo `src/features.py`:

- seleciona apenas as colunas de entrada do modelo;
- faz coerção dos tipos numéricos e categóricos;
- trata valores ausentes de forma compatível com o pipeline;
- remove registros sem alvo válido para treino.


In [ ]:
X, y = prepare_training_features(df, target_column=TARGET_COLUMN)

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

display(X.head())
display(y.head())

print("Distribuição final do target usado no treino:")
display(y.value_counts(dropna=False).sort_index().rename("qtd").to_frame())


### 5) Estratégia de separação treino e teste

Como este é um problema preditivo com componente temporal, a estratégia preferencial é:

- **treinar com anos anteriores**
- **testar no último ano disponível**

Isso reduz o risco de avaliação otimista demais.

Se não houver anos suficientes com alvo válido, usamos `train_test_split` estratificado como fallback.


In [ ]:
df_trainable = df.loc[y.index].copy()
anos_validos = (
    pd.to_numeric(df_trainable["ano_pede"], errors="coerce")
    .dropna()
    .astype(int)
    .sort_values()
    .unique()
)

use_temporal_split = len(anos_validos) >= 2

if use_temporal_split:
    test_year = int(anos_validos[-1])
    train_mask = pd.to_numeric(df_trainable["ano_pede"], errors="coerce") < test_year
    test_mask = pd.to_numeric(df_trainable["ano_pede"], errors="coerce") == test_year

    X_train = X.loc[train_mask].reset_index(drop=True)
    X_test = X.loc[test_mask].reset_index(drop=True)
    y_train = y.loc[train_mask].reset_index(drop=True)
    y_test = y.loc[test_mask].reset_index(drop=True)

    split_strategy = f"temporal (treino < {test_year}, teste = {test_year})"
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y if y.nunique() > 1 else None,
    )
    split_strategy = "aleatória estratificada (fallback)"

print("Estratégia usada:", split_strategy)
print("Treino:", X_train.shape, y_train.shape)
print("Teste :", X_test.shape, y_test.shape)

if "ano_pede" in df_trainable.columns:
    print("\nAnos no treino:")
    display(pd.Series(X_train["ano_pede"]).value_counts(dropna=False).sort_index().rename("qtd").to_frame())
    print("\nAnos no teste:")
    display(pd.Series(X_test["ano_pede"]).value_counts(dropna=False).sort_index().rename("qtd").to_frame())

print("\nDistribuição do target no treino:")
display(y_train.value_counts(dropna=False).sort_index().rename("qtd").to_frame())

print("\nDistribuição do target no teste:")
display(y_test.value_counts(dropna=False).sort_index().rename("qtd").to_frame())


### 6) Treinamento do pipeline

O pipeline é composto por:

- imputação de ausentes nas variáveis numéricas;
- padronização das variáveis numéricas;
- imputação e one-hot encoding das variáveis categóricas;
- regressão logística balanceada.


In [ ]:
pipeline = build_pipeline()
pipeline.fit(X_train, y_train)

print("Modelo treinado com sucesso.")
print(pipeline)


### 7) Predição e métricas de avaliação

As métricas principais usadas neste notebook são:

- **Accuracy**
- **Precision**
- **Recall**
- **F1-score**
- **ROC AUC** (quando disponível)

Como o objetivo é identificar alunos em risco, **recall** tende a ser especialmente importante, pois mede a capacidade de encontrar casos positivos.


In [ ]:
y_pred = pipeline.predict(X_test)

if hasattr(pipeline, "predict_proba"):
    y_score = pipeline.predict_proba(X_test)[:, 1]
else:
    y_score = None

metricas = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1_score": f1_score(y_test, y_pred, zero_division=0),
}

if y_score is not None and y_test.nunique() == 2:
    metricas["roc_auc"] = roc_auc_score(y_test, y_score)

metricas_df = pd.DataFrame(
    {"métrica": list(metricas.keys()), "valor": list(metricas.values())}
)
display(metricas_df)

print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Matriz de confusão:")
cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["Real 0", "Real 1"],
    columns=["Pred 0", "Pred 1"],
)
display(cm)


### 8) Amostra de predições


In [ ]:
resultado_teste = X_test.copy()
resultado_teste["y_real"] = y_test.values
resultado_teste["y_pred"] = y_pred

if y_score is not None:
    resultado_teste["prob_risco_1"] = y_score

display(resultado_teste.head(20))


### 9) Interpretação dos coeficientes

Como a regressão logística é interpretável, podemos analisar os coeficientes do modelo treinado.

- coeficientes **positivos** aumentam a chance da classe 1;
- coeficientes **negativos** reduzem a chance da classe 1.

Essa leitura ajuda a explicar o comportamento do modelo para públicos técnicos e de negócio.


In [ ]:
preprocessor = pipeline.named_steps["preprocessor"]
model = pipeline.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()
coeficientes = model.coef_.ravel()

coef_df = pd.DataFrame({
    "feature_transformada": feature_names,
    "coeficiente": coeficientes,
})
coef_df["abs_coeficiente"] = coef_df["coeficiente"].abs()

print("Top coeficientes positivos:")
display(coef_df.sort_values("coeficiente", ascending=False).head(15))

print("Top coeficientes negativos:")
display(coef_df.sort_values("coeficiente", ascending=True).head(15))


### 10) Salvamento dos artefatos

Ao final, o notebook salva:

- o pipeline treinado;
- os metadados necessários para inferência no app Streamlit.


In [ ]:
artifacts_dir = ROOT_DIR / "data" / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(pipeline, MODEL_FILE)
joblib.dump(
    {
        "feature_columns": FEATURE_COLUMNS,
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "target_column": TARGET_COLUMN,
        "metrics": {
            **metricas,
            "split_strategy": split_strategy,
            "train_rows": int(len(X_train)),
            "test_rows": int(len(X_test)),
        },
    },
    METADATA_FILE,
)

print("Modelo salvo em:", MODEL_FILE)
print("Metadados salvos em:", METADATA_FILE)


## Conclusão

Com a nova base de modelagem e o novo `src/features.py`, este notebook passa a treinar um modelo **preditivo de fato**, pois utiliza informações do ano atual para estimar o risco de defasagem no ano seguinte.

Os principais ganhos desta abordagem são:

- eliminação do vazamento de informação do alvo atual;
- maior aderência ao problema de negócio;
- avaliação mais coerente com cenário real, especialmente com separação temporal;
- pipeline pronto para serialização e deploy no Streamlit.
